# Intro

This notebook is for running the LLM as a judge scoring for the OSQs. 

Framework:
- Pull in the OSQ response data
- (Optional) list of judge models -- default only CXhatgpt5 or Chatgpt4o-mini
- List of Prompts
  - (Currently Omitted) Binary Correct/Incorrect
  - Scoring
    - Rubric 1 (from the MCQ -> OSQ conversion)
    - (Omitted) Rubric 2 (outside of recommended from conversion, e.g. from other sources)

# Configuration

In [10]:
# ================================================
# Phase 5 — Append-Only Judging (Resumable, Single JSONL, Progress)
# ================================================
from pathlib import Path
from datetime import datetime
from statistics import mean
from tqdm.auto import tqdm
from openai import OpenAI
import os, json

# -----------------------------
# CONFIG
# -----------------------------
TASK_NAME    = "sysengbench-osq"
JUDGE_MODEL  = "openai/gpt-5"
TEMPERATURE  = 0.0
MAX_TOKENS   = 2000
SAMPLE_N     = 3          # 0 = judge ALL samples; else judge first N (for quick tests)

# Paths (this notebook under: src/phase5_llm_as_a_judge/)
PHASE4_ROOT  = Path("../phase4_inference/downloaded_output") / TASK_NAME
# PHASE5_ROOT  = Path(".") / TASK_NAME  # mirror structure in phase5
PHASE5_ROOT  = Path(".") / f"{TASK_NAME}-llm-judge" # Append "-llm-judge" to keep Phase 5 artifacts separate and consistent
PHASE5_ROOT.mkdir(parents=True, exist_ok=True)

# OpenRouter client
client = OpenAI(
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url="https://openrouter.ai/api/v1"
)
print("OPENROUTER_API_KEY present:", bool(os.getenv("OPENROUTER_API_KEY")))

OPENROUTER_API_KEY present: True


## Optional Troubleshooting Directory

In [ ]:
# OPTIONAL TROUBLEHSOOTING CELL
from pathlib import Path
import json

p = Path("../phase4_inference/downloaded_output/sysengbench-osq/gemma3__27b")  # or whichever model dir
f = sorted(p.glob("samples_*.jsonl"))[-1]  # newest
print(f"File = {f}")

with open(f,"r",encoding="utf-8") as fh:
    for i,line in enumerate(fh):
        if i>2: break
        print(json.loads(line))


File = ..\phase4_inference\downloaded_output\sysengbench-osq\gemma3__27b\samples_sysengbench-osq_2025-10-01T04-26-37.597737.jsonl
{'doc_id': 0, 'doc': {'Question ID': 1, 'Tags': 'Introduction to risk', 'INCOSE Handbook Category': 'INCOSEHandbook/Systems Engineering Overview/System Concepts and Structures', 'question': 'What best describes the concept of uncertainty in systems engineering?', 'choiceA': 'The process of systematically improving and optimizing a system for efficiency.', 'choiceB': 'The condition where the outcomes of system functions are not predictable due to lack of information or variability.', 'choiceC': 'A method for analyzing the costs and benefits of a system over its lifecycle.', 'choiceD': 'The act of integrating different system components into a cohesive whole.', 'answer': 'B', 'label': 1, 'Justification': 'Uncertainty in systems engineering refers to the unpredictability of outcomes due to insufficient information or inherent variability within the system or it

# LLM-as-a-Judge Mega-Cell

In [ ]:
# Would be nice to have a cell that checks for how many samples have to be judged and have already been judged, to avoid re-judging them.

import json
import pandas as pd
from pathlib import Path

def build_judge_progress_matrix(
    phase4_root: str,
    phase5_root: str,
    task_name: str
) -> pd.DataFrame:
    """
    Build a matrix that summarizes judge progress for each model.

    Columns:
        model_name       – filesystem-safe model folder
        ollama_name      – restored with ':' instead of '__'
        judge_status     – not started | partial | complete
        judged_count     – # of judged OSQ responses so far
        total_samples    – total OSQ responses from Phase 4 output
        progress_fraction – judged_count / total_samples
    """

    p4 = Path(phase4_root)
    p5 = Path(phase5_root)

    if not p4.exists():
        raise FileNotFoundError(f"Phase 4 directory missing: {phase4_root}")
    if not p5.exists():
        raise FileNotFoundError(f"Phase 5 directory missing: {phase5_root}")

    model_folders = sorted([d.name for d in (p4 / task_name).iterdir() if d.is_dir()])

    rows = []

    for model in model_folders:
        model_p4_dir = p4 / task_name / model
        model_p5_dir = p5 / task_name / model

        # --------------------------------
        # 1. Count TOTAL samples (Phase 4)
        # --------------------------------

        # Find a samples_<task>_<ts>.jsonl file
        sample_files = [
            f for f in model_p4_dir.iterdir()
            if f.name.startswith(f"samples_{task_name}_") and f.suffix == ".jsonl"
        ]

        total_samples = 0
        if sample_files:
            # Usually only 1 samples file, but loop safely
            for sf in sample_files:
                with open(sf, "r", encoding="utf-8") as fh:
                    total_samples = sum(1 for _ in fh)
                break  # take first

        # --------------------------------
        # 2. Count JUDGED samples (Phase 5)
        # --------------------------------

        judged_count = 0

        if model_p5_dir.exists():
            judge_files = [
                f for f in model_p5_dir.iterdir()
                if f.name.startswith("judge_") and f.suffix == ".jsonl"
            ]

            for jf in judge_files:
                with open(jf, "r", encoding="utf-8") as fh:
                    judged_count += sum(1 for _ in fh)

        # --------------------------------
        # 3. Compute judge status
        # --------------------------------

        if total_samples == 0:
            judge_status = "not started"
        else:
            if judged_count == 0:
                judge_status = "not started"
            elif judged_count < total_samples:
                judge_status = "partial"
            else:
                judge_status = "complete"

        rows.append({
            "model_name": model,
            "ollama_name": model.replace("__", ":"),
            "judge_status": judge_status,
            "judged_count": judged_count,
            "total_samples": total_samples,
            "progress_fraction": (
                judged_count / total_samples if total_samples > 0 else 0.0
            ),
        })

    df = pd.DataFrame(rows)
    return df.sort_values("model_name")


In [8]:
TASK_NAME = "sysengbench-osq"

df = build_judge_progress_matrix(
    phase4_root= PHASE4_ROOT,
    phase5_root= PHASE5_ROOT,
    task_name=TASK_NAME
)

display(df)


FileNotFoundError: [WinError 3] The system cannot find the path specified: '..\\phase4_inference\\downloaded_output\\sysengbench-osq\\sysengbench-osq'

In [6]:
from pathlib import Path

p = Path("../phase4_inference/downloaded_output")
print("Resolved:", p.resolve())
print("Exists:", p.exists())
print("Contents:", [x.name for x in p.iterdir()])


Resolved: C:\Users\rabel\Desktop\dissertation\src\phase4_inference\downloaded_output
Exists: True
Contents: ['sysengbench', 'sysengbench-a', 'sysengbench-b', 'sysengbench-c', 'sysengbench-d', 'sysengbench-osq']


In [ ]:
from pathlib import Path
from collections import OrderedDict
from datetime import datetime
import json
from tqdm import tqdm

# -----------------------------
# Judge prompt (0–100 via five 0–20 dims) — braces escaped for .format()
# -----------------------------
JUDGE_PROMPT = """You are an expert systems engineering educator evaluating student responses to open-ended questions.

EVALUATION SCHEMA (each 0–20, total 100):
1. TECHNICAL ACCURACY — correctness of systems engineering concepts, terminology, and facts
2. CONCEPTUAL UNDERSTANDING — depth of comprehension of underlying SE principles
3. COMPLETENESS — coverage of key elements expected in the answer
4. CLARITY & ORGANIZATION — logical structure and clear communication
5. PROFESSIONAL RELEVANCE — connection to real-world SE practice and standards

QUESTION CONTEXT:
Original Question: {osq_question}
Expected Answer: {expected_answer}
Bloom's Level: {blooms_level}
SE Domain: {se_domain}

STUDENT RESPONSE TO EVALUATE:
{student_response}

INSTRUCTIONS:
Compare the response to the expected answer and criteria.
Return strict JSON in this format:
{{
  "technical_accuracy": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "conceptual_understanding": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "completeness": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "clarity_organization": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "professional_relevance": {{"score": <0–20>, "justification": "<1–2 sentences>"}},
  "overall_score": <0–100>,
  "overall_assessment": "<summary>",
  "key_strengths": "<strengths>",
  "improvement_areas": "<areas for improvement>"
}}
"""

# -----------------------------
# Helpers
# -----------------------------
def newest(path_iter):
    items = sorted([p for p in path_iter], key=lambda p: p.stat().st_mtime, reverse=True)
    return items[0] if items else None

def load_jsonl(path: Path):
    with open(path, "r", encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]

def safe_json(s: str):
    try:
        return json.loads(s)
    except Exception:
        return None

def extract_student_response(sample_row):
    """
    Option B confirmed: use resps[0].
    Some LM-Eval dumps store resps as [[text]] (list-of-list).
    Handle str or 1-deep nesting robustly.
    """
    resps = sample_row.get("resps")
    if not resps:
        return ""
    first = resps[0]
    if isinstance(first, list) and first:
        first = first[0]
    return first if isinstance(first, str) else ""

def extract_core_fields(sample_row):
    """Map Phase-4 LM-Eval row to the fields we judge on (with reasonable fallbacks only where specified)."""
    doc = sample_row.get("doc", {}) or {}
    osq_question    = doc.get("osq_prompt") or doc.get("question") or ""     # prefer osq_prompt; fallback to question
    expected_answer = doc.get("expected_answer", "")                         # E1 (no fallback to target)
    student_resp    = extract_student_response(sample_row)                   # Option B
    blooms_level    = doc.get("blooms_level", "N/A")
    se_domain       = doc.get("INCOSE Handbook Category", "General Systems Engineering")  # D1
    return {
        "osq_question": osq_question,
        "expected_answer": expected_answer,
        "student_response": student_resp,
        "blooms_level": blooms_level,
        "se_domain": se_domain,
    }

def ensure_alignment_or_die(existing_row, current_src_fields, sample_id):
    """If samples_*.jsonl already contains this sample_id, ensure its 'sample' snapshot matches Phase-4 source now."""
    snap = existing_row.get("sample", {})
    keys = ["osq_question", "expected_answer", "student_response", "blooms_level", "se_domain"]
    mismatches = [k for k in keys if (snap.get(k) or "") != (current_src_fields.get(k) or "")]
    if mismatches:
        raise RuntimeError(
            f"\n[ALIGNMENT ERROR] sample_id={sample_id} content changed in Phase-4.\n"
            f" Mismatched keys: {', '.join(mismatches)}\n"
            f" judged snapshot: {{k: snap.get(k) for k in keys}}\n"
            f" phase4 fields:   {{k: current_src_fields.get(k) for k in keys}}\n"
            f"Refusing to proceed. Investigate Phase-4 changes or freeze inputs."
        )

# -----------------------------
# Discover model directories
# -----------------------------
if not PHASE4_ROOT.exists():
    raise FileNotFoundError(f"Phase-4 task directory not found: {PHASE4_ROOT}")

model_dirs = [d for d in PHASE4_ROOT.iterdir() if d.is_dir()]
if not model_dirs:
    raise FileNotFoundError(f"No model directories under {PHASE4_ROOT}")

print(f"Discovered {len(model_dirs)} model directories under task '{TASK_NAME}'.")

# -----------------------------
# Main loop per model (single progress bar each)
# -----------------------------
for model_dir in model_dirs:
    model_name = model_dir.name
    src_samples = newest(model_dir.glob("samples_*.jsonl"))
    # src_results is not used anymore (no stats here)
    # src_results = newest(model_dir.glob("results_*.json"))

    if not src_samples:
        print(f"[skip] {model_name}: missing samples_*.jsonl")
        continue

    # Load Phase-4 samples (source of truth)
    source_rows = load_jsonl(src_samples)
    if SAMPLE_N > 0:
        source_rows = source_rows[:SAMPLE_N]
    total = len(source_rows)
    print(f"\nModel: {model_name} | Source: {src_samples.name} | Count: {total}")

    # Phase-5 output dir mirrors task/model
    out_dir = PHASE5_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)


    # ---------------------------------------------------------
    # Decide / locate final Phase-5 samples file (NO new names)
    # We take the Phase-4 samples filename verbatim and just append __<judge-model>.jsonl
    # ---------------------------------------------------------
    phase4_name = src_samples.name.replace(".jsonl","")  # full Phase-4 filename minus extension
    target_name = f"{phase4_name}__{JUDGE_MODEL.replace('/', '_')}.jsonl"

    out_dir = PHASE5_ROOT / model_name
    out_dir.mkdir(parents=True, exist_ok=True)

    final_samples_path = out_dir / target_name

    # If it already exists, resume. If not, create empty file.
    if final_samples_path.exists():
        print(f"[resume] Reusing existing samples file: {final_samples_path.name}")
    else:
        with open(final_samples_path, "w", encoding="utf-8") as _:
            pass
        print(f"[new] Creating samples file: {final_samples_path.name}")

    # Resume: read existing judged rows (by sample_id) from the final samples file
    done_ids = set()
    existing_by_id = {}
    existing_rows = load_jsonl(final_samples_path) if final_samples_path.exists() else []
    for row in existing_rows:
        sid = row.get("sample_id")
        if isinstance(sid, int):
            done_ids.add(sid)
            existing_by_id[sid] = row

    # Alignment check for already-judged IDs
    for sid in sorted(done_ids):
        if sid >= total:
            raise RuntimeError(f"{final_samples_path.name} has sample_id {sid} beyond current source length {total}.")
        src_fields_now = extract_core_fields(source_rows[sid])
        ensure_alignment_or_die(existing_by_id[sid], src_fields_now, sid)

    print(f"[progress] {model_name}: {len(done_ids)} already judged, {total - len(done_ids)} remaining.")

    # Judge remaining samples (append line-by-line) directly into the final samples file
    to_do = [i for i in range(total) if i not in done_ids]
    if not to_do:
        print(f"[done] {model_name}: nothing to judge.")
        continue

    with open(final_samples_path, "a", encoding="utf-8") as fout:
        for sid in tqdm(to_do, desc=f"Judging {model_name}", unit="sample"):
            src_fields = extract_core_fields(source_rows[sid])

            # Skip if core triad is missing
            missing = [k for k in ("osq_question","expected_answer","student_response")
                       if not isinstance(src_fields.get(k, ""), str) or not src_fields.get(k, "").strip()]
            if missing:
                record = {
                    "sample_id": sid,
                    "sample": src_fields,  # snapshot embedded
                    "judge": {
                        "fields": {
                            "technical_accuracy":      {"score": None, "justification": None},
                            "conceptual_understanding":{"score": None, "justification": None},
                            "completeness":            {"score": None, "justification": None},
                            "clarity_organization":    {"score": None, "justification": None},
                            "professional_relevance":  {"score": None, "justification": None},
                            "overall_score": None,
                            "overall_assessment": f"SKIPPED: missing fields {missing}",
                            "key_strengths": None,
                            "improvement_areas": None,
                        },
                        "prompt": None,
                        "raw_output": None,
                        "timestamp": datetime.now().isoformat(),
                        "meta": {
                            "judge_model": JUDGE_MODEL,
                            "temperature": TEMPERATURE,
                            "max_tokens": MAX_TOKENS,
                        }
                    }
                }
                fout.write(json.dumps(record) + "\n"); fout.flush()
                continue

            # Build prompt
            prompt = JUDGE_PROMPT.format(
                osq_question=src_fields["osq_question"],
                expected_answer=src_fields["expected_answer"],
                student_response=src_fields["student_response"],
                blooms_level=src_fields["blooms_level"],
                se_domain=src_fields["se_domain"],
            )

            # Call judge LLM
            raw = None
            parsed = None
            api_error = None
            try:
                completion = client.chat.completions.create(
                    model=JUDGE_MODEL,
                    temperature=TEMPERATURE,
                    max_tokens=MAX_TOKENS,
                    messages=[
                        {"role": "system", "content": "You are an expert evaluator in systems engineering education."},
                        {"role": "user", "content": prompt}
                    ]
                )
                raw = completion.choices[0].message.content.strip()
                parsed = safe_json(raw)
            except Exception as e:
                api_error = str(e)

            # Build judge fields (strict but resilient)
            fields = {
                "technical_accuracy":      {"score": None, "justification": None},
                "conceptual_understanding":{"score": None, "justification": None},
                "completeness":            {"score": None, "justification": None},
                "clarity_organization":    {"score": None, "justification": None},
                "professional_relevance":  {"score": None, "justification": None},
                "overall_score": None,
                "overall_assessment": None,
                "key_strengths": None,
                "improvement_areas": None,
            }
            if parsed is None:
                fields["overall_assessment"] = "ERROR: Invalid JSON response" + (f" ({api_error})" if api_error else "")
            else:
                def sget(d, key):
                    v = d.get(key)
                    return (v or {}).get("score") if isinstance(v, dict) else v
                fields["technical_accuracy"]       = {"score": sget(parsed,"technical_accuracy"),      "justification": (parsed.get("technical_accuracy") or {}).get("justification")}
                fields["conceptual_understanding"] = {"score": sget(parsed,"conceptual_understanding"),"justification": (parsed.get("conceptual_understanding") or {}).get("justification")}
                fields["completeness"]             = {"score": sget(parsed,"completeness"),            "justification": (parsed.get("completeness") or {}).get("justification")}
                fields["clarity_organization"]     = {"score": sget(parsed,"clarity_organization"),    "justification": (parsed.get("clarity_organization") or {}).get("justification")}
                fields["professional_relevance"]   = {"score": sget(parsed,"professional_relevance"),  "justification": (parsed.get("professional_relevance") or {}).get("justification")}
                fields["overall_score"]            = parsed.get("overall_score")
                fields["overall_assessment"]       = parsed.get("overall_assessment")
                fields["key_strengths"]            = parsed.get("key_strengths")
                fields["improvement_areas"]        = parsed.get("improvement_areas")

            # Append one JSON object per judged sample directly to the final samples file
            record = {
                "sample_id": sid,
                "sample": src_fields,  # embedded snapshot
                "judge": {
                    "fields": fields,
                    "prompt": prompt,
                    "raw_output": raw,
                    "timestamp": datetime.now().isoformat(),
                    "meta": {
                        "judge_model": JUDGE_MODEL,
                        "temperature": TEMPERATURE,
                        "max_tokens": MAX_TOKENS,
                    }
                }
            }
            fout.write(json.dumps(record) + "\n")
            fout.flush()

    print(f"[✓] Samples appended for {model_name}: {final_samples_path.name}")

print("\nAll models processed.")


Discovered 2 model directories under task 'sysengbench-osq'.

Model: gemma3__27b | Source: samples_sysengbench-osq_2025-10-01T04-26-37.597737.jsonl | Count: 3
[new] Creating samples file: samples_sysengbench-osq_2025-10-01T04-26-37.597737__openai_gpt-5.jsonl
[progress] gemma3__27b: 0 already judged, 3 remaining.


Judging gemma3__27b: 100%|██████████| 3/3 [00:53<00:00, 17.99s/sample]


[✓] Samples appended for gemma3__27b: samples_sysengbench-osq_2025-10-01T04-26-37.597737__openai_gpt-5.jsonl

Model: gemma3__4b | Source: samples_sysengbench-osq_2025-10-01T03-58-29.887651.jsonl | Count: 3
[new] Creating samples file: samples_sysengbench-osq_2025-10-01T03-58-29.887651__openai_gpt-5.jsonl
[progress] gemma3__4b: 0 already judged, 3 remaining.


Judging gemma3__4b: 100%|██████████| 3/3 [00:51<00:00, 17.18s/sample]

[✓] Samples appended for gemma3__4b: samples_sysengbench-osq_2025-10-01T03-58-29.887651__openai_gpt-5.jsonl

All models processed.


# Creating the results_ file for judging. this will require us to parse the output and create a new results *.json file